# Tutorial 05: Logging System

This tutorial demonstrates how to use the logging system in `amasedrp` to track preprocessing progress and capture warnings.

In [1]:
import os
import tempfile
import logging

import numpy as np
from astropy.io import fits

from amasedrp.imageprocessing import Image, image_preprocessing
from amasedrp.utils.logging import configure_logging

## Create temporary FITS files for the demo

We generate minimal dummy FITS files (bias, dark, flat, science) so that the notebook can be executed without external data.

In [2]:
tmp_dir = tempfile.mkdtemp(prefix="amase_logging_demo_")

shape = (100, 100)

# --- bias ---
bias_header = fits.Header()
bias_header["IMAGETYP"] = "bias"
bias_data = np.full(shape, 100, dtype=np.float32)
fits.PrimaryHDU(bias_data, header=bias_header).writeto(
    os.path.join(tmp_dir, "bias.fits"), overwrite=True
)

# --- dark ---
dark_header = fits.Header()
dark_header["IMAGETYP"] = "dark"
dark_header["EXPTIME"] = 60.0
dark_data = np.full(shape, 150, dtype=np.float32)
fits.PrimaryHDU(dark_data, header=dark_header).writeto(
    os.path.join(tmp_dir, "dark.fits"), overwrite=True
)

# --- pixel flat ---
flat_header = fits.Header()
flat_header["IMAGETYP"] = "flat"
flat_header["EXPTIME"] = 10.0
flat_data = np.full(shape, 200, dtype=np.float32)
fits.PrimaryHDU(flat_data, header=flat_header).writeto(
    os.path.join(tmp_dir, "flat.fits"), overwrite=True
)

# --- science ---
sci_header = fits.Header()
sci_header["IMAGETYP"] = "object"
sci_header["EXPTIME"] = 30.0
sci_data = np.full(shape, 250, dtype=np.float32)
fits.PrimaryHDU(sci_data, header=sci_header).writeto(
    os.path.join(tmp_dir, "science.fits"), overwrite=True
)

print("Temporary files created in:", tmp_dir)

Temporary files created in: /tmp/amase_logging_demo_xera60w7


## Method 1: Pass `log_file` to `image_preprocessing`

The easiest way to enable logging is to provide the `log_file` argument when calling the high-level `image_preprocessing` function.

The module logger will automatically write **INFO** and **WARNING** messages to the specified file.

In [3]:
log_path = os.path.join(tmp_dir, "pipeline.log")
output_path = os.path.join(tmp_dir, "calibrated.fits")

output = image_preprocessing(
    input_path=os.path.join(tmp_dir, "science.fits"),
    bias_path=os.path.join(tmp_dir, "bias.fits"),
    dark_path=os.path.join(tmp_dir, "dark.fits"),
    flat_path=os.path.join(tmp_dir, "flat.fits"),
    output_path=output_path,
    log_file=log_path,
)

print("Calibration done. Log file:", log_path)

Calibration done. Log file: /tmp/amase_logging_demo_xera60w7/pipeline.log


### Inspect the log file

In [4]:
with open(log_path, "r") as f:
    print(f.read())

2026-06-14 20:16:24,001 - amasedrp.imageprocessing.image_preprocessing - INFO - Applying bias subtraction...
2026-06-14 20:16:24,001 - amasedrp.imageprocessing.image_preprocessing - INFO - Bias subtraction applied.
2026-06-14 20:16:24,001 - amasedrp.imageprocessing.image_preprocessing - INFO - Applying dark subtraction...
2026-06-14 20:16:24,001 - amasedrp.imageprocessing.image_preprocessing - INFO - Dark subtraction applied.
2026-06-14 20:16:24,001 - amasedrp.imageprocessing.image_preprocessing - INFO - Applying pixel flat-field correction...
2026-06-14 20:16:24,001 - amasedrp.imageprocessing.image_preprocessing - INFO - Pixel flat-field correction applied.



## Method 2: Manually configure logging with `configure_logging`

For lower-level functions (e.g. `image_calibration`) or when you want to control the log level / format, use `amasedrp.utils.logging.configure_logging`.

It attaches a `FileHandler` to the root `amasedrp` logger, so messages from **any** `amasedrp` submodule are captured.

In [5]:
manual_log = os.path.join(tmp_dir, "manual.log")

configure_logging(
    log_file=manual_log,
    level=logging.INFO,
    fmt="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

### Trigger a warning and verify it appears in the log

We deliberately call `image_calibration` on an image that already has the `CALIBRAT` keyword to trigger a warning.

In [6]:
from amasedrp.imageprocessing import image_calibration

sci = Image.from_fits(os.path.join(tmp_dir, "science.fits"))
bias = Image.from_fits(os.path.join(tmp_dir, "bias.fits"))
dark = Image.from_fits(os.path.join(tmp_dir, "dark.fits"))
flat = Image.from_fits(os.path.join(tmp_dir, "flat.fits"))

# Set CALIBRAT to trigger the warning
sci.header["CALIBRAT"] = True

_ = image_calibration(
    input_image=sci,
    master_bias_image=bias,
    master_dark_image=dark,
    master_pixflat_image=flat,
    steps=("bias", "dark", "flat"),
)

In [7]:
with open(manual_log, "r") as f:
    print(f.read())

2026-06-14 20:16:24,087 [WARNING] amasedrp.imageprocessing.image_preprocessing: Input image already has CALIBRAT keyword; it may have been calibrated.
2026-06-14 20:16:24,087 [INFO] amasedrp.imageprocessing.image_preprocessing: Applying bias subtraction...
2026-06-14 20:16:24,088 [INFO] amasedrp.imageprocessing.image_preprocessing: Bias subtraction applied.
2026-06-14 20:16:24,088 [INFO] amasedrp.imageprocessing.image_preprocessing: Applying dark subtraction...
2026-06-14 20:16:24,088 [INFO] amasedrp.imageprocessing.image_preprocessing: Dark subtraction applied.
2026-06-14 20:16:24,088 [INFO] amasedrp.imageprocessing.image_preprocessing: Applying pixel flat-field correction...
2026-06-14 20:16:24,088 [INFO] amasedrp.imageprocessing.image_preprocessing: Pixel flat-field correction applied.



## Adjusting the log level

Only messages at or above the configured level are emitted.

- `logging.INFO` (default) captures INFO and WARNING.
- `logging.WARNING` suppresses INFO, keeping only warnings.
- `logging.DEBUG` also records DEBUG messages (if any are emitted).

In [8]:
warning_only_log = os.path.join(tmp_dir, "warning_only.log")

configure_logging(
    log_file=warning_only_log,
    level=logging.WARNING,
)

# Run the same calibration again
_ = image_calibration(
    input_image=sci,
    master_bias_image=bias,
    master_dark_image=dark,
    master_pixflat_image=flat,
    steps=("bias", "dark", "flat"),
)

with open(warning_only_log, "r") as f:
    print(f.read())

2026-06-14 20:16:24,150 - amasedrp.imageprocessing.image_preprocessing - WARNING - Input image already has CALIBRAT keyword; it may have been calibrated.



## Clean up temporary files

In [9]:
import shutil
shutil.rmtree(tmp_dir)
print("Cleaned up:", tmp_dir)

Cleaned up: /tmp/amase_logging_demo_xera60w7


## Summary

| Use case | How to enable |
| --- | --- |
| High-level pipeline | Pass `log_file="/path/to.log"` to `image_preprocessing()` |
| Low-level / custom scripts | Import `configure_logging` from `amasedrp.utils.logging` and call it before your code |
| Control verbosity | Set `level=logging.INFO` (default), `logging.WARNING`, or `logging.DEBUG` |
| Custom format | Pass `fmt="..."` to `configure_logging` |

All logs are written in **append mode**, so multiple runs accumulate in the same file.